# Cancer Type by Treatment Category — Stacked Bar (Figure S2A)

Top 10 cancer types (excluding Multiple Cancer Type Patient), stacked by mutually exclusive
treatment category.

**Cohort:** first line of therapy only. Each patient contributes exactly one row — their line-1
row, selected by `idxmin` on `lot` (not `.groupby().first()`, which takes the first non-null value
per column independently and can therefore assemble a `contains_*` treatment profile from a mix of
LOT-1 and later lines whenever LOT-1 has a null). Same line-1 convention as S2B and S2C.

Unlike S2C, no `lot_start` / `t_cutoff_lot` validity requirement is imposed: this is a cohort
composition panel with no time-to-event component, so censoring validity is not a precondition for
inclusion and adding it would drop patients from the reported N for a reason unrelated to what the
panel shows.

In [ ]:
# ---- Hard-fail if Arial isn't actually resolved (no silent fallback) ----
import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")

In [ ]:
%matplotlib inline
import re, os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.grid": False, "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 450,
})
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages

def standardize_mrn(mrn):
    if pd.isna(mrn):
        return None
    try:
        digits = re.findall(r"\d+", str(mrn).strip().strip("'\""))
        return str(int(digits[0])).zfill(8) if digits else None
    except (ValueError, TypeError):
        return None

def derive_treatment_category(row):
    def _on(col):
        return row.get(col, 0) in [1, True, "1", "True"]
    has_ctla4 = _on("contains_ctla4_immuno") or _on("contains_ctla4")
    has_pd1 = _on("contains_non_ctla4_immuno") or _on("contains_pd1")
    if _on("contains_immuno") or has_ctla4 or has_pd1:
        if has_pd1 and has_ctla4:
            return "PD-(L)1 + CTLA-4"
        if has_ctla4:
            return "CTLA-4"
        return "PD-(L)1"
    if _on("contains_chemo"):
        return "Chemotherapy"
    if _on("contains_hormone"):
        return "Hormone"
    if _on("contains_biologic"):
        return "Biologic"
    if _on("contains_targeted"):
        return "Targeted"
    return "Other"

In [ ]:
NOTEBOOK_DIR = os.getcwd()
FIGURES_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
DATA_DIR = os.path.join(FIGURES_DIR, 'figures_data', 'figure 2', 'data')
TOX_TABLE_DIR = os.path.join(DATA_DIR, 'OneDrive_1_8-7-2026')
RESULTS_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'results', 'supp', 'S2A_Cancer_Type'))
os.makedirs(RESULTS_DIR, exist_ok=True)

covars = pd.read_csv(os.path.join(TOX_TABLE_DIR, 'llm84k_pneumonitis_grade0_20260630.csv'), low_memory=False)
llm_patients = pd.read_csv(os.path.join(DATA_DIR, 'llm_calls_patient_level_84k.csv'), encoding="latin-1", low_memory=False)

covars["mrn"] = covars["mrn"].apply(standardize_mrn)
llm_patients["mrn"] = llm_patients["mrn"].apply(standardize_mrn)
covars = covars[covars["mrn"].notna()]
llm_patients = llm_patients[llm_patients["mrn"].notna()]

covars["lot"] = pd.to_numeric(covars["lot"], errors="coerce")
covars = covars[covars["lot"].notna()].copy()
covars = covars.sort_values(["mrn", "lot"])

# ---- First line of therapy: the patient's line-1 ROW, selected with idxmin. ----
# NOT .groupby().first() -- that takes the first non-null value per column independently,
# which can assemble a treatment profile from a mix of LOT-1 and later lines whenever
# LOT-1 has a null in any contains_* flag. Same line-1 convention as S2B and S2C.
n_patients_pre = covars["mrn"].nunique()
idx = covars.groupby("mrn")["lot"].idxmin()
patient_covars = covars.loc[idx].reset_index(drop=True)

assert len(patient_covars) == n_patients_pre, "line-1 selection changed the patient count"
assert patient_covars["mrn"].is_unique, "more than one line-1 row per patient"
print(f"{len(patient_covars):,} patients, one line-1 row each")
print(f"  line-1 LOT value distribution: {patient_covars['lot'].value_counts().sort_index().to_dict()}")

df = llm_patients.merge(patient_covars, on="mrn", how="inner")
assert df["mrn"].is_unique, "merge introduced duplicate patients"

df = df[df["cancer_type"].notna() & (df["cancer_type"] != "")].copy()
df = df[df["cancer_type"] != "Multiple Cancer Type Patient"]
df["treatment_category"] = df.apply(derive_treatment_category, axis=1)
print(f"Patients: {len(df):,}  (each contributing their line-1 treatment category)")

In [ ]:
# Build pivot: top 10 cancer types x treatment category
top_cancers = df["cancer_type"].value_counts().head(10).index.tolist()
df_top = df[df["cancer_type"].isin(top_cancers)].copy()

pivot = df_top.groupby(["cancer_type", "treatment_category"]).size().unstack(fill_value=0)
pivot = pivot.loc[top_cancers]  # preserve rank order

# Category stacking order: non-ICI at bottom, ICI at top
CATEGORY_ORDER = ["Chemotherapy", "Hormone", "Biologic",
                  "Targeted", "Other", "PD-(L)1", "CTLA-4", "PD-(L)1 + CTLA-4"]
present_cats = [c for c in CATEGORY_ORDER if c in pivot.columns]

CATEGORY_COLORS = {
    "Chemotherapy": "#E74C3C", "PD-(L)1": "#3498DB", "CTLA-4": "#1ABC9C",
    "PD-(L)1 + CTLA-4": "#117A65", "Hormone": "#9B59B6", "Biologic": "#2ECC71",
    "Targeted": "#E67E22", "Other": "#95A5A6",
}

# Legend order — same list used to build the legend handles below
LEGEND_ORDER = ["Chemotherapy", "PD-(L)1", "Hormone", "CTLA-4",
                "Targeted", "Biologic", "PD-(L)1 + CTLA-4", "Other"]

# Category counts for legend
cat_counts = df_top["treatment_category"].value_counts()
print("Category counts in top 10 cancers:")
for c in present_cats:
    print(f"  {c}: N={cat_counts.get(c, 0):,}")
print(f"  Overall: N={len(df_top):,}")

In [ ]:
def set_axes_position_inches(fig, ax, left_in, top_in, width_in, height_in):
    fw, fh = fig.get_size_inches()
    ax.set_position([
        left_in / fw,
        1 - (top_in + height_in) / fh,   # bottom, measured from figure bottom
        width_in / fw,
        height_in / fh,
    ])

FIG_WIDTH_IN    = 3.6
FIG_HEIGHT_IN   = 2.3
TOP_MARGIN_IN   = 0.05
RIGHT_MARGIN_IN = 0.05
# LEFT_MARGIN_IN / BOTTOM_MARGIN_IN / PLOT_WIDTH_IN / PLOT_HEIGHT_IN are
# computed automatically in the plotting cell after the labels are drawn,
# so they always fit regardless of label length/font/rotation.

In [ ]:
# Panel b
fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))
bottom = np.zeros(len(pivot))
for cat in present_cats:
    vals = pivot[cat].values
    ax.bar(range(len(pivot)), vals, bottom=bottom,
           color=CATEGORY_COLORS.get(cat, '#888'),
           edgecolor='white', linewidth=0.5, width=0.75)
    bottom += vals
legend_handles = [mpatches.Patch(color=CATEGORY_COLORS[cat], label=cat)
                   for cat in LEGEND_ORDER if cat in present_cats]
ax.legend(handles=legend_handles, loc='upper right', fontsize=5,
          ncol=2, framealpha=0.95, edgecolor='#ccc', fancybox=False,
          handlelength=1.0, handletextpad=0.4, columnspacing=0.8)
ax.set_xticks(range(len(pivot)))
ax.set_xticklabels(pivot.index, rotation=45, ha='right', rotation_mode='anchor', fontsize=6)
ax.set_ylabel('Number of Patients', fontsize=7)
ax.set_ylim(0, 7300)
ax.set_yticks(range(0, 8000, 1000))
ax.tick_params(axis='y', labelsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# --- place with a rough first guess, then measure real overflow and correct ---
guess_left, guess_bottom = 1.0, 0.9
set_axes_position_inches(fig, ax, left_in=guess_left, top_in=TOP_MARGIN_IN,
                          width_in=FIG_WIDTH_IN - guess_left - RIGHT_MARGIN_IN,
                          height_in=FIG_HEIGHT_IN - TOP_MARGIN_IN - guess_bottom)

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
tight_bbox = fig.get_tightbbox(renderer)  # in inches, figure-relative origin at bottom-left

overflow_left   = max(0, -tight_bbox.x0)
overflow_bottom = max(0, -tight_bbox.y0)
overflow_right  = max(0, tight_bbox.x1 - FIG_WIDTH_IN)
overflow_top    = max(0, tight_bbox.y1 - FIG_HEIGHT_IN)

LEFT_MARGIN_IN   = guess_left      + overflow_left   + 0.03   # +tiny safety pad
BOTTOM_MARGIN_IN = guess_bottom    + overflow_bottom + 0.03
RIGHT_MARGIN_IN  = RIGHT_MARGIN_IN + overflow_right  + 0.03
TOP_MARGIN_IN    = TOP_MARGIN_IN   + overflow_top    + 0.03
PLOT_WIDTH_IN    = FIG_WIDTH_IN  - LEFT_MARGIN_IN - RIGHT_MARGIN_IN
PLOT_HEIGHT_IN   = FIG_HEIGHT_IN - TOP_MARGIN_IN  - BOTTOM_MARGIN_IN

set_axes_position_inches(fig, ax, left_in=LEFT_MARGIN_IN, top_in=TOP_MARGIN_IN,
                          width_in=PLOT_WIDTH_IN, height_in=PLOT_HEIGHT_IN)

print(f"margins (in): left={LEFT_MARGIN_IN:.2f} right={RIGHT_MARGIN_IN:.2f} "
      f"top={TOP_MARGIN_IN:.2f} bottom={BOTTOM_MARGIN_IN:.2f}")
print(f"plot area (in): {PLOT_WIDTH_IN:.2f} x {PLOT_HEIGHT_IN:.2f}")

In [ ]:
# Save
os.makedirs(RESULTS_DIR, exist_ok=True)
with PdfPages(os.path.join(RESULTS_DIR, 'Cancer_Type_Treatment_S2A.pdf')) as pdf:
    pdf.savefig(fig, dpi=450)
plt.close(fig)
print("Saved: ../results/supp/S2A_Cancer_Type/Cancer_Type_Treatment_S2A.pdf")

In [ ]:
# Export underlying data to CSV
export_df = pivot[present_cats].copy()
export_df["Total"] = export_df.sum(axis=1)
export_df.to_csv(os.path.join(RESULTS_DIR, 'S2A_Cancer_Type.csv'))
print("Saved: ../results/supp/S2A_Cancer_Type/S2A_Cancer_Type.csv")